# 06 · Build the local full-enwiki BM25 index

Builds the offline knowledge corpus that kills the live-Wikipedia **429** (docs/enwiki_bm25.md):
`dump → wikiextractor → flatten → BM25 index`. The config (`configs/live.yaml`) is already pointed at
`data/corpus/enwiki/bm25_index`, so once this finishes you just **Restart runtime** and play.

> ⚠️ **Heavy, run once.** Full enwiki = ~22 GB download + a long wikiextractor pass, and the BM25 build
> wants **~32-64 GB RAM**. Standard Colab (~13 GB) will OOM on the FULL build → use a **High-RAM** runtime,
> or build the full index on a big machine and host the `bm25_index/` dir (Drive/HF) and just download it.
> **First do a SMOKE build** (cell 3, `MAX_DOCS=20000`) on any runtime to confirm the whole chain works.

## 1 · Clone / sync the repo

In [ ]:
import os
REPO_URL = 'https://github.com/SleepyEveryD/NLP.git'
BRANCH   = 'entertainment'
REPO_ROOT = '/content/NLP'
if not os.path.isdir(REPO_ROOT):
    !git clone -b {BRANCH} {REPO_URL} {REPO_ROOT}
else:
    !cd {REPO_ROOT} && git pull
os.chdir(REPO_ROOT)
print('cwd =', os.getcwd())

## 2 · Install build deps  (bm25s + wikiextractor)

In [ ]:
!pip -q install bm25s wikiextractor
import bm25s; print('bm25s', bm25s.__version__)

## 3 · Choose the build size

`MAX_DOCS` empty = **FULL** enwiki (needs High-RAM). Set e.g. `'20000'` for a **smoke** build first.

In [ ]:
import os
os.environ['MAX_DOCS'] = '20000'      # '' = FULL build. Start with a smoke number to verify the chain.
os.environ['OUT']      = 'data/corpus/enwiki/bm25_index'   # matches configs/live.yaml -> auto-active after Restart.
os.environ['PYTHONPATH'] = 'src'
print('MODE:', 'SMOKE (MAX_DOCS=' + os.environ['MAX_DOCS'] + ')' if os.environ['MAX_DOCS'] else 'FULL enwiki')
print('OUT :', os.environ['OUT'])

## 4 · Build  (download → extract → flatten → index)

Each step skips if its output already exists, so a re-run resumes. The FULL build is long (hours).

In [ ]:
!bash scripts/build_enwiki_bm25.sh

## 5 · Verify  (load the index, run a couple of questions)

In [ ]:
import sys; sys.path.insert(0, 'src')
from retrieval.bm25_retriever import BM25Retriever
from schemas import Question
r = BM25Retriever(index_dir=os.environ['OUT'], top_k=3)
for text, opts in [
    ("In which U.S. state was 'The Shawshank Redemption' primarily shot?",
     {'A': 'California', 'B': 'Ohio', 'C': 'Maine', 'D': 'New York'}),
    ("Which artist recorded the 1982 album 'Thriller'?",
     {'A': 'Prince', 'B': 'Michael Jackson', 'C': 'Stevie Wonder', 'D': 'Lionel Richie'}),
]:
    docs = r.retrieve(Question(qid='t', text=text, options=opts))
    print('Q:', text[:60])
    print('   top:', docs[0].doc_id if docs else '(none -> would fall back to live)')
    print('   evidence:', (docs[0].text[:160] if docs else ''))
    print()

## 6 · Enable it

`configs/live.yaml` already has `retrieval.bm25_index_path: data/corpus/enwiki/bm25_index` — so:

1. Make sure the index lives at that path (this notebook built it there).
2. **Runtime → Restart runtime**, then run `03_live_play.ipynb` / `05_news_test.ipynb` as usual.

Knowledge questions (Entertainment / History / Science / Philosophy) now hit the **local BM25 first**
(offline, no 429); only the few questions whose article is absent fall through to live Wikipedia.
**News is unchanged.**

**Persisting the index across sessions** (Colab wipes `/content`): copy `data/corpus/enwiki/bm25_index/`
(and the `articles.jsonl` it points to) to Google Drive or a HF dataset, and on the next session download
it back to the same path instead of rebuilding — loading is light, only the build is heavy.

```python
# e.g. save to Drive after a full build:
from google.colab import drive; drive.mount('/content/drive')
!cp -r data/corpus/enwiki /content/drive/MyDrive/enwiki_bm25
# next session: !cp -r /content/drive/MyDrive/enwiki_bm25 data/corpus/enwiki
```